# Task 04 — Multi-Turn Storage Ring Injection Validation Notebook

This notebook loads the storage ring lattice (`storage_ring_lattice_nkm.mat`), simulates multi-turn injection across four kicker models (NKM Off, Ideal Kicker, Linearized NKM, RADIA Fieldmap NKM), and evaluates physical multi-turn capture efficiency, loss accounting, and stored-beam perturbation.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure repository root is on sys.path
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.nkm.storage_ring_injection import (
    StorageRingInjectionConfig,
    load_storage_ring_injection_lattice,
    track_multiturn_injection,
    compute_multiturn_injection_metrics
)
from src.nkm.beam import generate_6d_beam
from src.nkm.kickmap import NKMKickMap2D

In [ ]:
# ── Simulation Configuration Summary ─────────────────────────────────────────
# Prints a table of all simulation parameters: their defaults (from config
# dataclasses) and any values reconfigured explicitly in this notebook.

import sys, os
from pathlib import Path

# Ensure repo root is on sys.path (already done in Cell 1; kept for safety)
_repo_root = Path('..').resolve()
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from src.nkm.storage_ring_injection import StorageRingInjectionConfig
from src.nkm.beam import generate_6d_beam

# ── Pull defaults from config dataclass ──────────────────────────────────────
_default_cfg = StorageRingInjectionConfig()

# ── Values explicitly set in this notebook ───────────────────────────────────
_notebook_n_particles  = 100
_notebook_n_turns      = 10
_notebook_inj_beta_x   = 10.0   # m   (injected beam)
_notebook_inj_beta_y   = 5.0    # m
_notebook_inj_emit_x   = 1e-7   # m·rad
_notebook_inj_emit_y   = 1e-8   # m·rad
_notebook_inj_x_offset = -0.016 # m   (septum position)
_notebook_sto_beta_x   = 10.0   # m   (stored beam)
_notebook_sto_beta_y   = 5.0    # m
_notebook_sto_emit_x   = 1e-8   # m·rad
_notebook_sto_emit_y   = 1e-9   # m·rad
_notebook_sto_x_offset = 0.0    # m
_notebook_seed         = 42
_notebook_kicker_models = ['off', 'ideal', 'linear', 'fieldmap']

# ── Table printer ─────────────────────────────────────────────────────────────
def _changed(default, notebook):
    return '⟵ reconfigured' if default != notebook else ''

rows = [
    # header
    ('Parameter', 'Default', 'This Notebook', 'Unit', 'Note'),
    # separator
    ('-' * 35, '-' * 18, '-' * 18, '-' * 10, '-' * 20),
    # Lattice / source data
    ('mat_filename',
     _default_cfg.mat_filename, _default_cfg.mat_filename, '—', ''),
    # Tracking
    ('n_particles',
     '(user defined)', _notebook_n_particles, 'count',
     _changed('(user defined)', _notebook_n_particles)),
    ('n_turns',
     10, _notebook_n_turns, 'turns',
     _changed(10, _notebook_n_turns)),
    ('kicker_models',
     '[fieldmap]', str(_notebook_kicker_models), '—', ''),
    # Apertures (from config)
    ('aperture_x (±)',
     f'{_default_cfg.aperture_x_m*1e3:.1f}',
     f'{_default_cfg.aperture_x_m*1e3:.1f}', 'mm', ''),
    ('aperture_y (±)',
     f'{_default_cfg.aperture_y_m*1e3:.1f}',
     f'{_default_cfg.aperture_y_m*1e3:.1f}', 'mm', ''),
    # Injected beam
    ('injected beta_x',
     '(user defined)', _notebook_inj_beta_x, 'm', ''),
    ('injected beta_y',
     '(user defined)', _notebook_inj_beta_y, 'm', ''),
    ('injected emit_x',
     '(user defined)', f'{_notebook_inj_emit_x:.0e}', 'm·rad', ''),
    ('injected emit_y',
     '(user defined)', f'{_notebook_inj_emit_y:.0e}', 'm·rad', ''),
    ('injected x_offset',
     f'{_default_cfg.septum_x_offset_m*1e3:.1f}',
     f'{_notebook_inj_x_offset*1e3:.1f}', 'mm', ''),
    # Stored beam
    ('stored beta_x',
     '(user defined)', _notebook_sto_beta_x, 'm', ''),
    ('stored beta_y',
     '(user defined)', _notebook_sto_beta_y, 'm', ''),
    ('stored emit_x',
     '(user defined)', f'{_notebook_sto_emit_x:.0e}', 'm·rad', ''),
    ('stored emit_y',
     '(user defined)', f'{_notebook_sto_emit_y:.0e}', 'm·rad', ''),
    ('stored x_offset',
     '(user defined)', f'{_notebook_sto_x_offset*1e3:.1f}', 'mm', ''),
    # Reproducibility
    ('random seed', 42, _notebook_seed, '—',
     _changed(42, _notebook_seed)),
]

col_w = [36, 19, 19, 11, 22]
sep   = '+' + '+'.join('-' * w for w in col_w) + '+'
head  = sep
print()
print('  SIMULATION CONFIGURATION — 02_multiturn_injection_validation')
print(sep)
for i, row in enumerate(rows):
    line = '|' + '|'.join(f' {str(v):<{col_w[j]-2}} ' for j, v in enumerate(row)) + '|'
    print(line)
    if i in (0, 1):
        print(sep)
print(sep)
print()


## 1. Load Storage Ring Lattice and Kick Map

In [ ]:
config = StorageRingInjectionConfig()
ring, nkm_idx = load_storage_ring_injection_lattice(config, mat_path=repo_root / config.mat_filename)
print(f"Loaded lattice: {len(ring)} elements, NKM inserted at index {nkm_idx}")

kickmap_obj = NKMKickMap2D(repo_root / "kickmap_file.txt")
print("Loaded 2D RADIA kick map.")

## 2. Multi-Turn Injection Model Comparison

In [ ]:
n_particles = 100
n_turns = 10

injected_beam = generate_6d_beam(
    n_particles=n_particles,
    beta_x=10.0, alpha_x=0.0, emit_x=1e-7,
    beta_y=5.0, alpha_y=0.0, emit_y=1e-8,
    x_offset=-0.016,
    seed=42
)

stored_beam = generate_6d_beam(
    n_particles=n_particles,
    beta_x=10.0, alpha_x=0.0, emit_x=1e-8,
    beta_y=5.0, alpha_y=0.0, emit_y=1e-9,
    x_offset=0.0,
    seed=42
)

models = ["off", "ideal", "linear", "fieldmap"]
results = {}

for model in models:
    inj_res = track_multiturn_injection(injected_beam, ring, n_turns=n_turns, kicker_model=model, kickmap_obj=kickmap_obj, config=config)
    stored_res = track_multiturn_injection(stored_beam, ring, n_turns=n_turns, kicker_model=model, kickmap_obj=kickmap_obj, config=config)
    metrics = compute_multiturn_injection_metrics(inj_res, stored_res, config)
    results[model] = metrics
    print(f"Model [{model:8s}]: Capture = {metrics['capture_efficiency']*100:.1f}%, Stored Osc = {metrics['stored_beam_centroid_oscillation_mm']:.4f} mm")